# Faruq-v3 — AF2 mechanism diagnostic

Membandingkan D0FT vs AF2 pada seed 42/123/2026 untuk memisahkan proposal/localization dan klasifikasi setelah localization. Tidak ada training dan test tidak dipulihkan. Setiap report disimpan ke Drive sehingga run dapat dilanjutkan setelah disconnect.

In [ ]:
BRANCH = 'agent/af2-continuation-confirmation'
DEVICE = '0'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO = Path('/content/coffee-bean-detection')
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
for name in list(sys.modules):
    if name == 'coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('GPU:',torch.cuda.get_device_name(0),'| BRANCH:',BRANCH)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
D0FT_REL = [
 'experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/weights/best.pt',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed123/weights/best.pt',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed2026/weights/best.pt',
]
AF2_REL = [
 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt',
 'experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed123/weights/best.pt',
 'experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed2026/weights/best.pt',
]
ARCHIVE_REL = 'bundles/faruq-development-v3-grouped.tar'
required = tuple([ARCHIVE_REL] + D0FT_REL + AF2_REL)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=required)
ARCHIVE = require_project_artifact(PROJECT_ROOT,ARCHIVE_REL)
D0FT = [require_project_artifact(PROJECT_ROOT,p) for p in D0FT_REL]
AF2 = [require_project_artifact(PROJECT_ROOT,p) for p in AF2_REL]
DATA = Path('/content/faruq-development-v3-grouped')
if not (DATA/'faruq_grouped_summary.json').is_file():
    if DATA.exists(): shutil.rmtree(DATA)
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'data.yaml').is_file() and not (DATA/'test').exists()
OUTPUT = PROJECT_ROOT/'experiments/faruq-v3-af2-mechanism-diagnostic-v1'
OUTPUT.mkdir(parents=True,exist_ok=True)
print('PROJECT:',PROJECT_ROOT); print('OUTPUT:',OUTPUT)

In [ ]:
command = [sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_mechanism_diagnostic',
 '--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),
 '--d0ft-checkpoints',*[str(p) for p in D0FT],
 '--af2-checkpoints',*[str(p) for p in AF2],
 '--output-root',str(OUTPUT),'--device',DEVICE]
LOG = OUTPUT/'diagnostic_run.log'
print('MENJALANKAN DIAGNOSTIC | log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process = subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
seen = None
while process.poll() is None:
    ready = len(list((OUTPUT/'reports').glob('*_diagnostic.json'))) if (OUTPUT/'reports').is_dir() else 0
    if ready != seen:
        print(f'Report selesai: {ready}/6',flush=True); seen = ready
    time.sleep(30)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-150:]))
    raise RuntimeError(f'Diagnostic gagal: {process.returncode}')
SUMMARY = OUTPUT/'af2_mechanism_diagnostic.json'
assert SUMMARY.is_file(),SUMMARY
print('SELESAI:',SUMMARY)

In [ ]:
import pandas as pd
from IPython.display import display
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
metrics = ['raw_proposal_accessibility','final_proposal_accessibility','matched_recall','conditional_top1_accuracy','localized_wrong_class_rate','proposal_miss_rate','correct_decision_recall']
rows = []
for metric in metrics:
    row = summary['aggregate'][metric]
    rows.append({'metric':metric,'D0FT':row['d0ft_mean'],'AF2':row['af2_mean'],'delta':row['delta_mean'],'improved_seeds':row['improved_seeds']})
display(pd.DataFrame(rows).style.format({'D0FT':'{:.2%}','AF2':'{:.2%}','delta':'{:+.2%}'}))
print('ATTRIBUTION:',summary['attribution'])
print('LOCALIZATION SUPPORTED:',summary['localization_supported'])
print('CLASSIFICATION SUPPORTED:',summary['classification_supported'])
print('CRITERIA:',summary['criteria'])
print('TRAINING:',summary['training_executed'],'| TEST:',summary['test_images_accessed'])
print('Kirim tabel dan attribution ini. Jangan training atau membuka test.')